In [2]:
import torch
import copy

In [4]:
#EMA
class EMA:
    #initialize
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        #initialize weights of shadow
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()
    #update
    def update(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad:
                assert name in self.shadow
                #update param
                new_avg = (
                    self.decay * self.shadow[name] + (1.0 - self.decay) * param.data
                )
                self.shadow[name] = new_avg.clone()
    #apply shadow
    def apply_shadow(self, model):
        #save params
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data = self.shadow[name]
    #restore
    def restore(self, model):
        #restore original params
        for name, param in model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]
        self.backup = {}